# 035-loss-arms-rerun: audit

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from uavloc.features.segmentation import load_segmenter
from uavloc.onboard.package import sha256_of
from uavloc.segmentation.eval import load_val_sets
import uavloc.segmentation.village_seg as vs
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "035-loss-arms-rerun"
SOURCE_EXP = "033-class-loss-in-system"
ARMS = ["control", "building", "structure", "noise"]
CONTROL = "control"
REFERENCE_EXP = "027-sequence-filters"
REFERENCE_RUN = "027.12"
REFERENCE_SEGMENTER = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"   # relative to src/
REFERENCE_HEAD = "experiments/023-learned-correlation-head/artifacts/head-023.3.pt"
SEG_ARCHITECTURE = "lraspp"
DEVICE = "cpu"
SEG_METRICS = ["building_missed_share_mean", "track_skeleton_recall_mean", "track_skeleton_precision_mean",
               "separable_merge_rate_mean", "miou_mean"]
FILTER = "particles"
Q_POS_M_PER_SQRT_S = 2.0
SHA_CHARS = 12

In [ ]:
SOURCE_ARTIFACTS = experiment_dir(SOURCE_EXP, create=False) / "artifacts"
OUT = experiment_dir(EXP) / "artifacts"
FIGURES = experiment_dir(EXP) / "figures"
OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# Data

In [ ]:
SEGMENTERS = {**{arm: f"experiments/{SOURCE_EXP}/artifacts/segmenter-{arm}/model-bn-satellite.pt" for arm in ARMS},
              REFERENCE_RUN: REFERENCE_SEGMENTER}
HEADS = {**{arm: f"experiments/{SOURCE_EXP}/artifacts/head-{arm}.pt" for arm in ARMS}, REFERENCE_RUN: REFERENCE_HEAD}

In [ ]:
files = pd.DataFrame({name: {"segmenter": SEGMENTERS[name], "segmenter_sha256": sha256_of(REPO_ROOT / SEGMENTERS[name])[:SHA_CHARS],
                              "head": HEADS[name], "head_sha256": sha256_of(REPO_ROOT / HEADS[name])[:SHA_CHARS]}
                      for name in SEGMENTERS}).T
files

In [ ]:
training = pd.DataFrame({arm: {k: json.loads((SOURCE_ARTIFACTS / f"segmenter-{arm}" / "config.json").read_text())[k]
                                for k in ("arm", "loss", "seed")} for arm in ARMS}).T
training

In [ ]:
recorded_runs = pd.DataFrame([{"run": r["run"], "loss_arm": r["config"]["loss_arm"],
                               "seg_checkpoint_sat": r["config"]["correlator"]["seg_checkpoint_sat"],
                               "seg_checkpoint_drone": r["config"]["correlator"]["seg_checkpoint_drone"],
                               "head": r["config"]["head"], "temperature": r["config"]["filter"]["temperature"],
                               "mean_end_error_m": r["metrics"]["mean_end_error_m"]} for r in load_runs(SOURCE_EXP)])
recorded_runs

In [ ]:
reference_config = next(r for r in load_runs(REFERENCE_EXP) if r["run"] == REFERENCE_RUN)["config"]
{k: reference_config["correlator"][k] for k in ("seg_checkpoint_sat", "seg_checkpoint_drone")} | {
    "likelihood": reference_config["likelihood"], "temperature": reference_config["filter"]["temperature"]}

In [ ]:
val_sets = load_val_sets(vs.SplitConfig().dataset)
{name: len(images) for name, images in val_sets.items()}

# Model

In [ ]:
def score_checkpoint(checkpoint: str) -> dict[str, float]:
    """The 021 satellite val metrics of a saved satellite-BN segmenter, relative path under src/."""
    return vs.val_metrics(load_segmenter(REPO_ROOT / checkpoint, DEVICE, SEG_ARCHITECTURE), val_sets, DEVICE)

In [ ]:
def tuning_curve(arm: str) -> pd.Series:
    """Tuning-villages mean end error per temperature, from the arm's 033 tuning shards."""
    shards = pd.concat([pd.read_csv(p) for p in sorted((SOURCE_ARTIFACTS / f"tuning-{arm}").glob("shard-[0-9].csv"))])
    shards = shards[(shards["filter"] == FILTER) & (shards["q_pos_m_per_sqrt_s"] == Q_POS_M_PER_SQRT_S)]
    return shards.set_index("temperature")["mean_end_error_m"].sort_index()

# Run

In [ ]:
scored = pd.DataFrame({name: score_checkpoint(path) for name, path in tqdm(SEGMENTERS.items(), desc="segmenters")}).T[SEG_METRICS]
scored.round(4)

In [ ]:
recorded = pd.DataFrame({arm: json.loads((SOURCE_ARTIFACTS / f"segmenter-{arm}" / "metrics.json").read_text())
                         for arm in ARMS}).T[SEG_METRICS].astype(float)
(scored.loc[ARMS] - recorded).abs().max()

In [ ]:
fingerprint = pd.DataFrame({"missed_buildings_vs_control": scored["building_missed_share_mean"] - scored.loc[CONTROL, "building_missed_share_mean"],
                            "track_recall_vs_control": scored["track_skeleton_recall_mean"] - scored.loc[CONTROL, "track_skeleton_recall_mean"]})
fingerprint.round(3)

In [ ]:
tuning = pd.DataFrame({arm: tuning_curve(arm) for arm in ARMS})
tuning.round(3)

In [ ]:
chosen_temperature = tuning.idxmin()
chosen_temperature

In [ ]:
results = {"files": files.to_dict("index"), "training": training.to_dict("index"),
           "recorded_runs": recorded_runs.to_dict("records"), "reference": reference_config["correlator"],
           "scored": scored.to_dict("index"), "recorded": recorded.to_dict("index"),
           "max_abs_scored_minus_recorded": (scored.loc[ARMS] - recorded).abs().max().to_dict(),
           "fingerprint": fingerprint.to_dict("index"), "tuning": {str(k): v for k, v in tuning.to_dict().items()},
           "chosen_temperature": chosen_temperature.to_dict()}
(OUT / "audit.json").write_text(json.dumps(results, indent=1, default=str))

# Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, metric in zip(axes, ["building_missed_share_mean", "track_skeleton_recall_mean", "miou_mean"]):
    scored[metric].plot.bar(ax=ax, color="#2a78d6")
    ax.set_title(metric, fontsize=9)
fig.suptitle("saved segmenters, re-scored on the satellite val sets (CPU)")
fig.tight_layout()
fig.savefig(FIGURES / "audit-segmentation.png", dpi=150)